# Pipeline for Stage 4
- braai, DeepStreaks, stamp classifier, light classifier.


## Rung 0: Load the handoff

In [ ]:
import sys, os
import numpy as np 
from astropy.table import Table
from astropy.io import fits

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # notebooks/ -> repo root
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
# paths
CATALOG_PATH = os.path.join(PROJECT_ROOT, "ztfdata",
    "ztf_own_pipeline_data_testing_small_scale", "catalog",
    "ztf_20180322273264_000468_zr_c03_o_q2_stage3_catalog.ecsv")
CUTOUTS_DIR = os.path.join(PROJECT_ROOT, "ztfdata",
    "ztf_own_pipeline_data_testing_small_scale", "cutouts")
DIFF_PATH = os.path.join(PROJECT_ROOT, "ztfdata", "difference",
    "ztf_20180322273264_000468_zr_c03_o_q2_official_diff.fits")

# load + verify
catalog = Table.read(CATALOG_PATH)
print(f"catalog: {len(catalog)} sources")
print(f"cutouts dir exists: {os.path.isdir(CUTOUTS_DIR)}")
print(f"diff FITS exists:   {os.path.exists(DIFF_PATH)}")

# the elongation split (our streak-branch cutoff)
n_elong = int((catalog["elongation"] > 1.5).sum())
print(f"{n_elong} of {len(catalog)} have elongation > 1.5 → streak branch")

## Rung 1: braai gate
load once, loop rows, P(real) + pass/fail

In [ ]:
from ztf_classification.braai_realbogus import load_braai
from ztf_classification.pipeline import run_braai_gate

model = load_braai(os.path.join(PROJECT_ROOT, "braai", "models"))
results = run_braai_gate(catalog, CUTOUTS_DIR, model)

passers = [r for r in results if r["braai_pass"]]
print(f"braai: {len(passers)} of {len(results)} passed (P(real) ≥ 0.5)")
for r in passers:
    print(f"  {r['row_id']}: P(real)={r['braai_p_real']:.4f}")


In [ ]:
from alerce.core import Alerce
from ztf_classification.pipeline import run_pathB_type

client = Alerce()
results = run_pathB_type(results, client)

for r in results:
    if r["braai_pass"]:
        if r["pathB_matched"]:
            print(f"  {r['row_id']}: MATCH oid={r['pathB_oid']} → {r['pathB_type']} (P={r['pathB_prob']:.3f})")
        else:
            print(f"  {r['row_id']}: NO-MATCH (→ Path A in Rung 3)")


In [ ]:
from ztf_classification.pathA_metadata import read_fits_header_fields, load_psfcat
from ztf_classification.pipeline import run_pathA_type

SCI = os.path.join(PROJECT_ROOT, "ztfdata", "sci", "2018", "0322", "273264",
                   "ztf_20180322273264_000468_zr_c03_o_q2_sciimg.fits")
PSFCAT = SCI.replace("sciimg.fits", "psfcat.fits")
CKPT = os.path.join(PROJECT_ROOT, "stamp_classifier_model", "results", "staging_model",
    "DeepHits_EntropyRegBeta0.5000_batch64_lr0.00100_droprate0.5000_inputsize21_filtersize5_0_20200708-160759",
    "checkpoints")
NORM = CKPT.replace("/checkpoints", "") + "/feature_norm_stats.pkl"

hdr_fields = read_fits_header_fields(SCI)
psfcat = load_psfcat(PSFCAT)
results = run_pathA_type(results, hdr_fields, psfcat, CKPT, NORM, CUTOUTS_DIR)

for r in results:
    if r.get("pathA_type"):
        print(f"  {r['row_id']}: Path A → {r['pathA_type']} (uncalibrated ranking)")


In [ ]:
import matplotlib.pyplot as plt
from ztf_classification.pipeline import build_streak_batch

diff_data = fits.getdata(DIFF_PATH)
streak_batch, streak_ids = build_streak_batch(results, catalog, diff_data)
print(f"streak batch: {streak_batch.shape}  ({len(streak_ids)} elongated stamps)")

# visual checkpoint: dark streak/source on a light-gray sky?
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, i in zip(axes, range(3)):
    ax.imshow(streak_batch[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(streak_ids[i]); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
from ztf_classification.pipeline import run_streak_subprocess

SCRATCH = os.path.join(PROJECT_ROOT, "ztfdata", "scratch")
streak_results = run_streak_subprocess(streak_batch, streak_ids, SCRATCH)

# merge routes back onto the detection records, by row_id
route_by_id = {rid: d["route"] for rid, d in streak_results.items()}
for r in results:
    r["streak_route"] = route_by_id.get(r["row_id"])   # None if not elongated

from collections import Counter
print(Counter(d["route"] for d in streak_results.values()))


In [ ]:
from ztf_classification.pipeline import build_verdict_table
from collections import Counter

OUT = os.path.join(PROJECT_ROOT, "ztfdata",
    "ztf_own_pipeline_data_testing_small_scale", "stage4_pipeline_verdicts.ecsv")
verdicts = build_verdict_table(results, OUT)

print(f"wrote {len(verdicts)} verdicts → {OUT}\n")
print("verdict distribution:", dict(Counter(verdicts["final_verdict"])))
print()
# show the interesting ones: the 3 braai passers
for row in verdicts:
    if row["braai_pass"]:
        print(f"  {row['row_id']}: {row['final_verdict']}  "
              f"(type={row['point_source_type']}, path={row['type_path']})")
